# 03c — Variables instrumentales reales (Hausman-Nevo)

**Proyecto:** Análisis de Elasticidad de Demanda en Delivery de Comida (v2)
**Fase:** 3c — Identificación causal con variables instrumentales

**Contexto importante:** en una fase anterior, un documento generado con
ayuda de IA describió un análisis de variables instrumentales completo
(2SLS, test de Hansen J, F-test de instrumentos débiles) que **nunca se
ejecutó** — con instrumentos ("costos de insumos", "distancia logística")
que no existen en este dataset. Ese documento fue descartado (ver bitácora,
sección 5.0). Este notebook implementa una estrategia de instrumentos
**real**, construida únicamente con columnas que sí existen.

Objetivos:
1. Construir 2 instrumentos reales para `checkout_price` (estrategia tipo
   Hausman-Nevo)
2. Estimar el modelo por 2SLS
3. Evaluar la fuerza de los instrumentos (F de primera etapa)
4. Test de sobre-identificación (Hansen J) — posible porque tenemos 2
   instrumentos para 1 variable endógena
5. Documentar honestamente el supuesto de exclusión (no es comprobable
   directamente, se argumenta cualitativamente)


In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

from modeling import (
    construir_instrumentos,
    modelo_iv_2sls,
    test_sobreidentificacion_hansen_j,
)

pd.set_option('display.max_columns', None)

## 1. Cargar datos

In [2]:
df = pd.read_csv('../data/processed/demanda_limpia.csv')
df.shape

(456548, 20)

## 2. Construcción de instrumentos

**Instrumento 1 — `log_base_price`:** el precio de referencia
administrativo. Se actualiza con menor frecuencia que `checkout_price`
(ver hallazgo de la Fase 2), por lo que está fuertemente correlacionado
con el precio final (relevancia) pero es plausible que no responda a
shocks de demanda semanales específicos de un centro (candidato razonable
a cumplir exclusión).

**Instrumento 2 — `log_precio_prom_otros_centros`:** el precio promedio de
la MISMA comida en OTROS centros esa MISMA semana (leave-one-out mean),
excluyendo la observación propia. Técnica estándar en organización
industrial (Hausman, 1996; Nevo, 2001): los precios en otros mercados
responden a shocks de costos compartidos a nivel de toda la cadena, pero
no deberían responder a shocks de demanda específicos del centro que
estamos evaluando.

**Limitación esperada:** las comidas ofrecidas en un solo centro no tienen
"otros centros" con qué construir el instrumento 2 -- esas filas quedan
con NaN y se excluyen del análisis IV (se cuantifica cuántas a continuación).

In [3]:
df_instrumentado = construir_instrumentos(df)

n_total = len(df_instrumentado)
n_sin_instrumento = df_instrumentado['log_precio_prom_otros_centros'].isna().sum()
print(f'Filas totales: {n_total:,}')
print(f'Filas sin "otros centros" para el instrumento 2: {n_sin_instrumento:,} '
      f'({n_sin_instrumento/n_total*100:.2f}%)')
print(f'Filas utilizables para IV: {n_total - n_sin_instrumento:,}')

Filas totales: 456,548
Filas sin "otros centros" para el instrumento 2: 4 (0.00%)
Filas utilizables para IV: 456,544


In [4]:
df_instrumentado[['checkout_price', 'base_price', 'precio_prom_otros_centros']].describe()

,checkout_price,base_price,precio_prom_otros_centros
count,456548.000000,456548.000000,456544.000000
mean,332.238933,354.156627,332.240566
std,152.939723,160.715914,151.324355
min,2.970000,55.350000,95.580492
25%,228.950000,243.500000,226.132105
50%,296.820000,310.460000,298.420451
75%,445.230000,458.870000,429.284857
max,866.270000,866.270000,749.141111


## 3. Correlación instrumentos-precio (chequeo previo de relevancia)

In [5]:
correlaciones = df_instrumentado[[
    'log_checkout_price', 'log_base_price', 'log_precio_prom_otros_centros'
]].corr()
correlaciones

,log_checkout_price,log_base_price,log_precio_prom_otros_centros
log_checkout_price,1.000000,0.955685,0.985788
log_base_price,0.955685,1.000000,0.953284
log_precio_prom_otros_centros,0.985788,0.953284,1.000000


## 4. Estimación 2SLS (especificación principal, con los mismos controles que el OLS)

**Ecuación estructural:**
`log_num_orders ~ 1 + emailer_for_promotion + homepage_featured + hubo_aumento + C(category) + C(center_type) + [log_checkout_price ~ log_base_price + log_precio_prom_otros_centros]`

La notación `[endógena ~ instrumentos]` le indica a `linearmodels` que `log_checkout_price` es la
variable endógena. Se reportan EE robustos y, como inferencia principal, EE agrupados por `center_id`.

In [6]:
resultado_iv, df_iv_usado = modelo_iv_2sls(df_instrumentado)
print(resultado_iv)

                          IV-2SLS Estimation Summary                          
Dep. Variable:         log_num_orders   R-squared:                      0.5304
Estimator:                    IV-2SLS   Adj. R-squared:                 0.5304
No. Observations:              456544   F-statistic:                 7.426e+05
Date:                Thu, Oct 22 2026   P-value (F-stat)                0.0000
Time:                        02:22:46   Distribution:                 chi2(19)
Cov. Estimator:                robust                                         
                                                                              
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------
Intercept                       9.5319     0.0298     319.61     0.0000      9.4

## 5. Fuerza de los instrumentos (primera etapa)

Con una sola variable endógena, el R² parcial de Shea coincide con el R² parcial conjunto de los
instrumentos excluidos. Para 2 instrumentos y 1 endógena, Stock y Yogo (2005) tabulan valores
críticos de *tamaño* (10%: 19.93); los de sesgo relativo requieren al menos 3 instrumentos.

In [7]:
print(resultado_iv.first_stage)

          First Stage Estimation Results          
                                log_checkout_price
--------------------------------------------------
R-squared                                   0.9767
Partial R-squared                           0.9226
Shea's R-squared                            0.9226
Partial F-statistic                       2.86e+06
P-value (Partial F-stat)                    0.0000
Partial F-stat Distn                       chi2(2)
===============================        ===========
Intercept                                  -0.2159
                                         (-65.374)
emailer_for_promotion                      -0.0536
                                         (-92.038)
homepage_featured                          -0.0136
                                         (-28.732)
hubo_aumento                                0.0312
                                          (137.82)
C(category)[T.Biryani]                     -0.0196
                               

In [8]:
import statsmodels.formula.api as smf
CTRL = 'emailer_for_promotion + homepage_featured + hubo_aumento + C(category) + C(center_type)'
fs = smf.ols(f'log_checkout_price ~ log_base_price + log_precio_prom_otros_centros + {CTRL}', df_iv_usado).fit()
fs0 = smf.ols(f'log_checkout_price ~ {CTRL}', df_iv_usado).fit()
print(f'R2 parcial de Shea (1 endógena): {1 - fs.ssr / fs0.ssr:.4f}')
fs_cl = smf.ols(fs.model.formula, df_iv_usado).fit(cov_type='cluster', cov_kwds={'groups': df_iv_usado['center_id']})
print('F de instrumentos excluidos (agrupado por center_id):',
      fs_cl.f_test('log_base_price = 0, log_precio_prom_otros_centros = 0'))

R2 parcial de Shea (1 endógena): 0.9226


F de instrumentos excluidos (agrupado por center_id): <F test: F=218018.54326895677, p=1.466054080009653e-143, df_denom=76, df_num=2>


## 6. Comparación OLS vs. 2SLS

| Modelo | Elasticidad | EE agrupado |
|---|---|---|
| OLS con controles (Fase 3) | -0.759 | 0.043 (entidad) |
| Panel con efectos fijos (Fase 3b) | -1.5985 | 0.024 (entidad) |
| 2SLS con controles (este notebook) | ver celda siguiente | ver celda siguiente (center_id) |

In [9]:
from linearmodels.iv import IV2SLS, IVGMM
from modeling import modelo_log_log_controles
elasticidad_ols = modelo_log_log_controles(df).params['log_checkout_price']
formula_iv = ('log_num_orders ~ 1 + emailer_for_promotion + homepage_featured + hubo_aumento '
              '+ C(category) + C(center_type) + [log_checkout_price ~ log_base_price + log_precio_prom_otros_centros]')
iv_cluster = IV2SLS.from_formula(formula_iv, df_iv_usado).fit(cov_type='clustered', clusters=df_iv_usado['center_id'])
elasticidad_iv = resultado_iv.params['log_checkout_price']
ee_iv = iv_cluster.std_errors['log_checkout_price']
print(f'Elasticidad (2SLS): {elasticidad_iv:.4f}')
print(f'EE robusto: {resultado_iv.std_errors["log_checkout_price"]:.4f} | EE agrupado (center_id): {ee_iv:.4f}')
print(f'IC 95% agrupado: [{elasticidad_iv - 1.96 * ee_iv:.3f}, {elasticidad_iv + 1.96 * ee_iv:.3f}]')
print(f'Elasticidad OLS con controles (Fase 3): {elasticidad_ols:.4f}')

Elasticidad (2SLS): -0.8414
EE robusto: 0.0056 | EE agrupado (center_id): 0.0502
IC 95% agrupado: [-0.940, -0.743]
Elasticidad OLS con controles (Fase 3): -0.7590


## 7. Test de sobre-identificación (Hansen J)

Con 2 instrumentos para 1 variable endógena, hay 1 grado de libertad de
sobre-identificación, lo que permite este test.

**H0:** todos los instrumentos son válidos (no correlacionados con el
error estructural). Un p-valor bajo (< 0.05) rechazaría H0, sugiriendo que
al menos uno de los dos instrumentos viola el supuesto de exclusión.

**Importante:** este test NO puede confirmar que los instrumentos son
válidos -- solo puede detectar inconsistencia *entre* ellos. Si ambos
instrumentos violan la exclusión de manera similar, el test no lo detecta.

In [10]:
j_stat = test_sobreidentificacion_hansen_j(df_iv_usado)
print('Hansen J (ponderación robusta a heterocedasticidad):', j_stat)
j_cl = IVGMM.from_formula(formula_iv, df_iv_usado, weight_type='clustered', clusters=df_iv_usado['center_id']).fit(
    cov_type='clustered', clusters=df_iv_usado['center_id']).j_stat
print('Hansen J (ponderación agrupada por center_id):', j_cl)

Hansen J (ponderación robusta a heterocedasticidad): H0: Expected moment conditions are equal to 0
Statistic: 33.2801
P-value: 0.0000
Distributed: chi2(1)


Hansen J (ponderación agrupada por center_id): H0: Expected moment conditions are equal to 0
Statistic: 0.9228
P-value: 0.3367
Distributed: chi2(1)


## 8. Discusión honesta del supuesto de exclusión

*(Completar esta celda en el informe final, no solo en el notebook, con el
resultado real del test de Hansen J de arriba)*

El supuesto de exclusión -- que ninguno de los dos instrumentos afecta la
demanda salvo a través de su efecto sobre `checkout_price` -- **no es
comprobable directamente**, ni siquiera con un test de sobre-identificación
que "pase": ese test solo detecta inconsistencia entre instrumentos, no
confirma validez absoluta. Se declara como supuesto identificador
explícito, con el siguiente argumento cualitativo:
- `base_price` es una variable fijada administrativamente, con baja
  frecuencia de actualización, por lo que es poco probable que responda a
  shocks de demanda de una semana particular
- El precio promedio en otros centros esa semana refleja principalmente
  condiciones de costos compartidas a nivel de cadena, no shocks de
  demanda local del centro evaluado -- aunque no se puede descartar que
  existan shocks de demanda correlacionados entre centros cercanos
  geográficamente (por ejemplo, un evento regional), lo cual sería una
  amenaza real a la exclusión que se documenta como limitación.

## 9. Qué NO se pudo instrumentar (documentado explícitamente)

A diferencia del documento fabricado, que inventaba "costos de insumos" y
"distancia logística" como instrumentos, este dataset **no contiene** esas
variables. Tampoco contiene un identificador de restaurante individual (solo
`center_id` y `meal_id`), por lo que no es posible construir instrumentos a
ese nivel de granularidad. Se documenta como limitación real, no se
inventan las variables faltantes.

## 10. Guardar resultados de la Fase 3c

In [11]:
import json

resumen_3c = {
    'n_filas_usadas_iv': int(len(df_iv_usado)),
    'n_filas_excluidas_sin_instrumento': int(n_sin_instrumento),
    'elasticidad_2sls': float(elasticidad_iv),
    'ee_2sls': float(ee_iv),
    'hansen_j_estadistico': float(j_stat.stat),
    'hansen_j_pvalor': float(j_stat.pval),
}

with open('../reports/resultados_fase_3c.json', 'w') as f:
    json.dump(resumen_3c, f, indent=2, default=str)

print('Guardado en reports/resultados_fase_3c.json')

Guardado en reports/resultados_fase_3c.json


## 11. Chequeo de robustez — IV justamente identificado

Dado que el test de Hansen J (sección 7) rechazó la validez conjunta de
los 2 instrumentos, y que ambos están muy correlacionados entre sí (0.953,
ver sección 3), se corre una versión con **un solo instrumento**
(`log_base_price`). Al quedar justamente identificado (1 instrumento para
1 variable endógena), no hay grados de libertad de sobre-identificación —
no existe Hansen J que pueda fallar. Sirve para verificar si la elasticidad
se mantiene cercana a la del modelo con 2 instrumentos.

In [12]:
from modeling import modelo_iv_justamente_identificado

resultado_iv_justo = modelo_iv_justamente_identificado(
    df_instrumentado, instrumento='log_base_price'
)
print(resultado_iv_justo)

elasticidad_iv_justo = resultado_iv_justo.params['log_checkout_price']
print(f'\nElasticidad (IV justamente identificado, solo base_price): {elasticidad_iv_justo:.4f}')
print(f'Elasticidad (2SLS con 2 instrumentos): {elasticidad_iv:.4f}')
print(f'Diferencia: {abs(elasticidad_iv_justo - elasticidad_iv):.4f}')

                          IV-2SLS Estimation Summary                          
Dep. Variable:         log_num_orders   R-squared:                      0.5302
Estimator:                    IV-2SLS   Adj. R-squared:                 0.5302
No. Observations:              456548   F-statistic:                 7.455e+05
Date:                Thu, Oct 22 2026   P-value (F-stat)                0.0000
Time:                        02:23:32   Distribution:                 chi2(19)
Cov. Estimator:                robust                                         
                                                                              
                                      Parameter Estimates                                      
                             Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------------------
Intercept                       9.5990     0.0308     311.19     0.0000      9.5

**Lectura:** las estimaciones justamente identificadas (solo `base_price`: -0.8545; solo
precio de otros centros: -0.8369) son cercanas a la de 2 instrumentos (-0.8414) en relación con
sus EE agrupados (~0.05). Esto es compatible tanto con instrumentos válidos como con una violación
de la exclusión compartida por ambos, por lo que el IV se presenta como análisis de sensibilidad.

## Próximos pasos
- Repetir Fase 4 (validación de supuestos) sobre el modelo seleccionado como principal entre OLS/panel/IV
- Reescribir el informe LaTeX con todos los resultados reales de 3b y 3c, eliminando cualquier resto de contenido fabricado
- Fase 6: capa SQL + BI